# Aprende GitHub Actions paso a paso

**Basado en:** `Github_Actions_2026.pptx` (Máster en Big Data & Cloud — VII Edición, Ignacio Reyes Vázquez)

Este notebook es una guía práctica y progresiva para entender **GitHub Actions**. Cada sección tiene:

1. **Teoría corta** (qué es y para qué sirve)
2. **Celdas ejecutables** que puedes correr aquí mismo (`!comando` usa la shell)
3. **Inspección de los workflows reales** del repo `.github/workflows/`
4. **Retos** para que practiques

> 💡 Cada celda de código se ejecuta con `Shift+Enter`. El `!` delante del comando indica que es un comando de shell (bash/cmd).

## 0. Preparación: ¿dónde estamos?

Antes de nada, verificamos que estamos dentro del repo y que git funciona correctamente.

In [ ]:
import os, subprocess, pathlib, textwrap, json

# Nos movemos a la raíz del repo (carpeta padre de este notebook)
REPO = pathlib.Path.cwd()
print('Carpeta actual:', REPO)
print('¿Es un repo git?:', (REPO / '.git').exists() or (REPO.parent / '.git').exists())

In [ ]:
# Estado actual del repositorio git
!git status

In [ ]:
# Últimos 5 commits para ver el historial
!git log --oneline -5

In [ ]:
# Rama actual (debería ser main o una rama de trabajo)
!git branch --show-current

## 1. ¿Qué es GitHub Actions?

**GitHub Actions** es una plataforma de **CI/CD** (Integración Continua / Despliegue Continuo) integrada directamente en GitHub.

- **CI — Continuous Integration**: cada vez que cambias código (push / PR), se revisa automáticamente (tests, linters, build...).
- **CD — Continuous Deployment**: si todo está OK, el código se publica automáticamente (a un servidor, a Docker Hub, a AWS...).

### La idea clave

```
Algo ocurre (Event)  →  se dispara un Workflow  →  que lanza uno o varios Jobs
                                                    →  cada Job corre en un Runner (VM)
                                                    →  cada Job ejecuta varios Steps
                                                    →  cada Step corre un script o una Action
```

### Componentes

| Componente | Qué es |
|---|---|
| **Workflow** | Proceso automatizado definido en un YAML dentro de `.github/workflows/` |
| **Event**    | Actividad que dispara el workflow (push, PR, manual, cron...) |
| **Job**      | Conjunto de pasos que se ejecutan en un mismo runner |
| **Runner**   | Máquina virtual (Ubuntu, Windows, macOS) donde corre el job |
| **Step**     | Paso individual: un comando `run:` o una acción `uses:` |
| **Action**   | Módulo reutilizable de la comunidad (ej: `actions/checkout`) |

In [ ]:
# Veamos qué workflows hay ya definidos en este repo
workflows_dir = pathlib.Path('.github/workflows')
for f in sorted(workflows_dir.iterdir()):
    size = f.stat().st_size
    print(f'  {f.name:<40} {size:>5} bytes')

## 2. Anatomía de un workflow — Tu primer YAML

Un workflow **mínimo** necesita 4 cosas:

1. `name:` → nombre visible en la pestaña *Actions*
2. `on:` → qué **evento** lo dispara
3. `jobs:` → uno o más **trabajos**
4. Dentro de cada job: `runs-on:` + `steps:`

Vamos a ver el ejemplo más simple del repo.

In [ ]:
# Leemos el workflow más básico del repo
print(pathlib.Path('.github/workflows/1.single_step.yaml').read_text())

### Explicación línea por línea

```yaml
name: First Action - Single   # ← nombre del workflow

on:
  workflow_dispatch:          # ← se ejecuta MANUALMENTE desde la UI de GitHub

jobs:
  hello-world:                # ← id del job (único dentro del workflow)
    runs-on: ubuntu-latest    # ← VM Ubuntu que GitHub nos presta gratis
    steps:
    - name: Fist Step         # ← nombre legible del step
      run: echo "Hello World" # ← comando de shell que se ejecuta
```

### 🎯 Ejecútalo en tu máquina

No necesitas GitHub para ver qué hace el `run:`: es **bash normal**.

In [ ]:
# Simulamos el step localmente: esto es lo mismo que el runner ejecutará
!echo "Hello World"

## 3. Múltiples steps dentro de un job

Los steps **se ejecutan en orden**, uno detrás de otro, en el **mismo runner**. Si uno falla, los siguientes no se ejecutan (por defecto).

In [ ]:
print(pathlib.Path('.github/workflows/2.multiple_step.yaml').read_text())

In [ ]:
# Simulamos la ejecución en local, step a step
print('--- Step 1: Fist Step ---')
!echo "Hello World"
print('--- Step 2: Second Step ---')
!echo "Bye World"

## 4. Múltiples jobs — paralelo vs. secuencial

### 🔀 Por defecto: los jobs corren en **PARALELO**
Cada job corre en su propio runner nuevo (no comparten sistema de archivos).

### ➡️ Con `needs:` los obligas a correr en **SECUENCIA**
`job_B` `needs: job_A` → `job_B` no empieza hasta que `job_A` termine OK.

In [ ]:
print('### PARALELO (3.multiple_job.yaml) ###')
print(pathlib.Path('.github/workflows/3.multiple_job.yaml').read_text())
print()
print('### SECUENCIAL (4.multiple_job_lineal.yaml) ###')
print(pathlib.Path('.github/workflows/4.multiple_job_lineal.yaml').read_text())

👉 Fíjate en la línea `needs: hello-world` del job `bye-world` en el segundo ejemplo. Eso es lo que fuerza la dependencia.

```
Paralelo:        hello-world  ─┐
                                ├─→ (los dos a la vez)
                 bye-world    ─┘

Secuencial:      hello-world ──→ bye-world
```

## 5. Eventos — ¿qué dispara un workflow?

Los eventos más comunes:

| Evento | Cuándo se dispara |
|---|---|
| `workflow_dispatch` | Manualmente desde la UI (botón *Run workflow*) |
| `push`              | Al hacer push a una rama |
| `pull_request`      | Al abrir/sincronizar/reabrir un PR |
| `schedule`          | Cron (ej: `'0 9 * * 1'` = lunes 9:00) |
| `release`           | Al publicar una release |

Puedes filtrar: sólo ciertas ramas, sólo ciertos paths, sólo ciertos tipos de PR...

In [ ]:
print('### Evento PUSH a main ###')
print(pathlib.Path('.github/workflows/5.event_commit.yaml').read_text())
print()
print('### Evento PULL REQUEST ###')
print(pathlib.Path('.github/workflows/6.event_pr.yaml').read_text())

## 6. Inputs — pedir datos al usuario

Cuando el evento es `workflow_dispatch` o `workflow_call`, puedes definir **inputs**: campos que el usuario rellena antes de lanzar el workflow.

Los inputs tienen propiedades:
- `description:` texto de ayuda
- `required:` true/false
- `default:` valor por defecto
- `type:` `string`, `boolean`, `number`, `choice`, `environment`

Se acceden desde los steps con: `${{ inputs.mi_input }}` o `${{ github.event.inputs.mi_input }}`.

In [ ]:
print('### third.yaml — input obligatorio ###')
print(pathlib.Path('.github/workflows/third.yaml').read_text())
print()
print('### 7.multiple_job_argument.yaml — dos inputs + condicional ###')
print(pathlib.Path('.github/workflows/7.multiple_job_argument.yaml').read_text())

### 🔍 Fíjate en la clave: `if:`

```yaml
bye-world:
  if: ${{ github.event.inputs.debug_mode == 'true' }}
```

El job `bye-world` **sólo se ejecuta** si el usuario marcó la casilla `debug_mode`. Es una forma muy potente de hacer workflows condicionales.

### Simulación local de un input

In [ ]:
# Simulamos lo que GitHub hace con la variable de input
nombre_usuario = 'Francisco'   # ← lo que el usuario escribió en la UI
debug_mode = 'true'            # ← checkbox marcado

os.environ['THENAME'] = nombre_usuario
os.environ['DEBUG_MODE'] = debug_mode

!echo "Welcome to Miami Beach, $THENAME!!"
!echo "Debug mode está: $DEBUG_MODE"

## 7. Contexts — acceder a información del workflow

Los **contexts** son objetos con propiedades a los que accedes con la sintaxis `${{ contexto.propiedad }}`.

Los más usados:

| Context | Contiene | Ejemplo |
|---|---|---|
| `github` | info del evento, repo, actor, ref... | `${{ github.actor }}` |
| `inputs` | valores que pasó el usuario | `${{ inputs.thename }}` |
| `env`    | variables de entorno del workflow | `${{ env.MI_VAR }}` |
| `secrets`| datos sensibles (enmascarados) | `${{ secrets.API_KEY }}` |
| `steps`  | outputs de steps anteriores | `${{ steps.mi_step.outputs.algo }}` |
| `matrix` | valor actual de la matriz | `${{ matrix.os }}` |
| `runner` | info del runner (OS, arch...) | `${{ runner.os }}` |

In [ ]:
# Ejemplo de uso de contexts con un snippet equivalente en shell
ejemplo = '''
steps:
  - name: Mostrar info del evento
    run: |
      echo "Usuario que disparó: ${{ github.actor }}"
      echo "Rama/ref: ${{ github.ref }}"
      echo "Evento: ${{ github.event_name }}"
      echo "SHA del commit: ${{ github.sha }}"
      echo "Runner OS: ${{ runner.os }}"
'''
print(ejemplo)

In [ ]:
# Información equivalente que obtienes en local desde git:
!git config user.name
!git rev-parse HEAD
!git branch --show-current

## 8. Variables de entorno (`env`) y Secretos (`secrets`)

### `env:` — valores **no sensibles** reutilizables
Se pueden definir a nivel de workflow, job o step.

```yaml
env:
  GLOBAL_USER: "estudiante_edem"

jobs:
  demo:
    runs-on: ubuntu-latest
    steps:
      - run: echo "El usuario es $GLOBAL_USER"
```

### `secrets.` — datos **sensibles** (contraseñas, tokens, keys)
Se configuran en GitHub: *Settings → Secrets and variables → Actions*.
**GitHub los enmascara automáticamente** en los logs (salen como `***`).

```yaml
- run: curl -H "Authorization: Bearer ${{ secrets.API_TOKEN }}" https://api.example.com
```

⚠️ **NUNCA** pongas contraseñas directamente en el YAML.

In [ ]:
# Simulación: usamos variables de entorno locales como si fueran de GitHub
os.environ['GLOBAL_USER'] = 'estudiante_edem'
os.environ['FAKE_PASSWORD'] = 'superSecreto123'

!echo "El usuario es $GLOBAL_USER"
# En GitHub, esto imprimiría: ***
!echo "Mi password es $FAKE_PASSWORD   (en GitHub se enmascararía)"

## 9. Outputs — pasar datos entre steps y jobs

Un step puede **exportar un valor** para que otro step/job lo use después.

### Dentro del mismo job (step → step)
```yaml
steps:
  - id: get-time                       # ← necesitamos un id
    name: Captura la hora
    run: |
      curTime=$(date)
      echo "currenttime=$curTime" >> $GITHUB_OUTPUT   # ← así se exporta

  - run: echo "La hora era ${{ steps.get-time.outputs.currenttime }}"
```

### Entre jobs (job → job)
Necesitas declarar `outputs:` en el job que exporta y usar `needs.` en el que consume.

In [ ]:
# Simulación local del mecanismo $GITHUB_OUTPUT
import tempfile, datetime

github_output_file = tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.txt')
github_output_path = github_output_file.name
github_output_file.close()

# Step 1: escribe su output
current_time = datetime.datetime.now().isoformat()
with open(github_output_path, 'a') as f:
    f.write(f'currenttime={current_time}\n')

# Step 2: lee el output del step anterior
with open(github_output_path) as f:
    contenido = f.read()
print('Contenido de $GITHUB_OUTPUT:')
print(contenido)
print(f'Leído por step 2: {current_time}')

### Ejemplo completo de outputs entre jobs

```yaml
jobs:
  generar:
    runs-on: ubuntu-latest
    outputs:
      numero: ${{ steps.rand.outputs.n }}          # ← expone output
    steps:
      - id: rand
        run: echo "n=$RANDOM" >> $GITHUB_OUTPUT

  consumir:
    runs-on: ubuntu-latest
    needs: generar                                  # ← depende
    steps:
      - run: echo "Recibí ${{ needs.generar.outputs.numero }}"
```

## 10. Matrix Strategy — probar varias combinaciones a la vez

`strategy.matrix` ejecuta el **mismo job N veces** con distintos valores. Típico para probar:
- Varias versiones de Python / Node / Java
- Varios sistemas operativos (Ubuntu / Windows / macOS)

```yaml
jobs:
  test:
    strategy:
      matrix:
        os: [ubuntu-latest, windows-latest, macos-latest]
        python: ['3.10', '3.11', '3.12']
    runs-on: ${{ matrix.os }}
    steps:
      - uses: actions/setup-python@v4
        with:
          python-version: ${{ matrix.python }}
      - run: python --version
```

Esto genera **9 jobs** (3 OS × 3 versiones Python) que corren **en paralelo**.

In [ ]:
# Visualizamos la matriz que se generaría
sistemas = ['ubuntu-latest', 'windows-latest', 'macos-latest']
pythons  = ['3.10', '3.11', '3.12']

combinaciones = [(o, p) for o in sistemas for p in pythons]
print(f'Total de jobs que se lanzarían: {len(combinaciones)}\n')
for i, (o, p) in enumerate(combinaciones, 1):
    print(f'  Job #{i}: OS={o:<16} Python={p}')

## 11. Uses — reutilizar Actions de la comunidad

En lugar de `run:` (un script), puedes usar `uses:` para llamar a una **Action** ya hecha.

Las más comunes:

| Action | Para qué sirve |
|---|---|
| `actions/checkout@v4` | Descarga el código del repo al runner |
| `actions/setup-python@v4` | Instala Python |
| `actions/setup-node@v4` | Instala Node.js |
| `actions/upload-artifact@v4` | Sube archivos como artefactos |
| `actions/download-artifact@v4` | Descarga artefactos |
| `docker/build-push-action@v6` | Build de imagen Docker |

⚠️ **Siempre fija una versión** (`@v4`), nunca uses `@main`.

In [ ]:
# El workflow de Python del repo es un ejemplo realista con varias Actions
print(pathlib.Path('.github/workflows/8.python_pr.yaml').read_text())

### Desglose del workflow de Python

Este workflow es un **pipeline de CI completo**:

1. **`on:`** se dispara en PR, push a `main` y manual.
2. **Job `linting`** → descarga código, instala Python 3.9, instala dependencias, corre tests (`unittest`), linter (`flake8`), seguridad (`bandit`), valida `Dockerfile` (`hadolint`).
3. **Job `docker`** → depende de `linting` (`needs:`) y construye la imagen Docker.

Si `linting` falla, `docker` **nunca se ejecuta** → así proteges la rama principal.

## 12. Artifacts — compartir archivos entre jobs / descargar del runner

Los runners son **desechables**: al terminar un job, todo lo que esté ahí se pierde. Si quieres:
- **pasar un archivo grande** a otro job (un binario, un reporte, un `tfplan`...)
- **descargar el resultado** desde GitHub después

...usa **artifacts**.

```yaml
- uses: actions/upload-artifact@v4
  with:
    name: mi-reporte
    path: ./reporte.txt

# En otro job:
- uses: actions/download-artifact@v4
  with:
    name: mi-reporte
```

Ejemplo real: el workflow de Terraform del repo usa artifacts para pasar el `tfplan` entre `plan` y `apply`.

In [ ]:
# Fíjate en upload-artifact / download-artifact en el workflow de Terraform
contenido = pathlib.Path('.github/workflows/9.terraform.yaml').read_text()
for i, linea in enumerate(contenido.splitlines(), 1):
    if 'artifact' in linea.lower() or 'tfplan' in linea.lower():
        print(f'{i:>3}: {linea}')

## 13. Custom Actions — crear tu propia Action

Hay 3 tipos:

| Tipo | Cómo funciona |
|---|---|
| **Docker** | El código corre dentro de un contenedor (útil para entornos específicos) |
| **JavaScript** | Corre en Node.js usando `@actions/core`. Muy rápida, ideal para acciones simples |
| **Composite** | Agrupa varios steps en una Action reutilizable (sin código extra) |

### Action Composite (la más sencilla)
Se guarda en `.github/actions/mi-action/action.yaml`:

```yaml
name: 'Saludar'
description: 'Saluda a alguien'
inputs:
  nombre:
    description: 'Nombre de la persona'
    required: true
runs:
  using: 'composite'
  steps:
    - run: echo "Hola ${{ inputs.nombre }}!"
      shell: bash
```

Y la llamas desde un workflow: `uses: ./.github/actions/mi-action`

## 14. 🧪 Mini-laboratorio: crear tu propio workflow

Vamos a **generar un nuevo workflow YAML desde Python** y validar su sintaxis.

> ⚠️ La celda siguiente **no modifica nada** en `.github/workflows/` por defecto. Cambia `GUARDAR = True` si quieres que lo escriba al disco.

In [ ]:
GUARDAR = False   # ← ponlo en True para escribir el archivo

mi_workflow = '''\
name: Mi primer workflow personalizado

on:
  workflow_dispatch:
    inputs:
      nombre:
        description: '¿Cómo te llamas?'
        required: true
        default: 'Alumno'
      entusiasmo:
        description: 'Nivel de entusiasmo'
        type: choice
        options: [bajo, medio, alto]
        default: medio

jobs:
  saludar:
    runs-on: ubuntu-latest
    steps:
      - name: Saludo básico
        run: echo "Hola ${{ inputs.nombre }}!"

      - name: Saludo con entusiasmo
        if: ${{ inputs.entusiasmo == 'alto' }}
        run: echo "¡¡¡HOLAAAA ${{ inputs.nombre }}!!! 🎉"

      - name: Mostrar contexto de GitHub
        run: |
          echo "Actor:   ${{ github.actor }}"
          echo "Repo:    ${{ github.repository }}"
          echo "Rama:    ${{ github.ref_name }}"
          echo "Evento:  ${{ github.event_name }}"
'''

print(mi_workflow)

if GUARDAR:
    destino = pathlib.Path('.github/workflows/mi_workflow.yaml')
    destino.write_text(mi_workflow)
    print(f'\n✅ Guardado en: {destino}')
else:
    print('\n(no guardado: cambia GUARDAR = True si quieres escribirlo)')

In [ ]:
# Validamos que el YAML es sintácticamente correcto
try:
    import yaml
    parsed = yaml.safe_load(mi_workflow)
    print('✅ YAML válido')
    print('Jobs definidos:', list(parsed.get('jobs', {}).keys()))
    print('Inputs:', list(parsed['on']['workflow_dispatch']['inputs'].keys()))
except ImportError:
    print('(instala pyyaml: pip install pyyaml)')
except Exception as e:
    print(f'❌ YAML inválido: {e}')

## 15. Inspeccionar git — cómo ve GitHub Actions tu repo

Las Actions no funcionan sin git. Vamos a ver qué información **tiene disponible** GitHub Actions cuando se ejecuta (y cómo sacarla tú localmente).

In [ ]:
# Info equivalente a los contexts de github.* que vimos antes
print('🔹 github.sha          →', end=' '); subprocess.run(['git', 'rev-parse', 'HEAD'])
print('🔹 github.ref_name     →', end=' '); subprocess.run(['git', 'branch', '--show-current'])
print('🔹 github.actor (tu user) →', end=' '); subprocess.run(['git', 'config', 'user.name'])
print('🔹 Último mensaje commit →')
subprocess.run(['git', 'log', '-1', '--pretty=format:   %h %s (%an)'])

In [ ]:
# Archivos que han cambiado en el último commit (muy útil para workflows con paths)
!git diff --name-only HEAD~1 HEAD

In [ ]:
# Historial completo de cambios en la carpeta .github/workflows/
!git log --oneline -- .github/workflows/

## 16. 🏆 Retos — ponte a prueba

Estos ejercicios vienen del `EXERCISES.md` del repo. Intenta escribir cada workflow tú mismo antes de mirar soluciones:

1. **Variables globales**: un workflow con `env` a nivel de workflow usado en dos jobs distintos.
2. **Secretos**: simula un login con `${{ secrets.DB_PASSWORD }}` y comprueba que aparece como `***` en los logs.
3. **Matrix**: corre tests en Python 3.11 y 3.12 simultáneamente.
4. **Artefactos**: Job 1 genera `saludo.txt`, Job 2 lo descarga y lo imprime.
5. **Filtros**: workflow que sólo se active en `push` a rama `develop` y sólo si cambian archivos en `docs/`.
6. **Validar JSON**: usar una action de la comunidad para validar un `data.json`.
7. **Outputs entre jobs**: Job A genera `$RANDOM`, Job B lo recibe y lo imprime.
8. **Cron**: workflow que corra todos los lunes a las 9:00 AM (`'0 9 * * 1'`).
9. **Servicios**: levanta un Postgres como service container y verifica la conexión.
10. **Aprobación manual**: job `Deploy` que sólo corre tras `Build` OK y si el evento es manual.

### 💡 Ayuda para el reto #3 (matrix)

```yaml
jobs:
  test:
    strategy:
      matrix:
        python: ['3.11', '3.12']
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v4
        with: { python-version: '${{ matrix.python }}' }
      - run: python -m unittest discover -s code/python -p "test_*.py"
```

## 17. 📋 Cheat-sheet final

Imprime esta celda y tenla a mano.

In [ ]:
cheat = '''
╔══════════════════════════════════════════════════════════════════════╗
║                 GITHUB ACTIONS — HOJA DE TRUCOS                      ║
╚══════════════════════════════════════════════════════════════════════╝

UBICACIÓN:    .github/workflows/*.yaml

ESQUELETO MÍNIMO:
  name: Mi Workflow
  on: [push]                       # o workflow_dispatch, pull_request, schedule
  jobs:
    mi_job:
      runs-on: ubuntu-latest       # o windows-latest, macos-latest
      steps:
        - uses: actions/checkout@v4
        - run: echo "hola"

EVENTOS MÁS USADOS:
  workflow_dispatch   → manual (botón Run workflow)
  push                → al hacer push
  pull_request        → al abrir/actualizar PR
  schedule            → cron, ej: "0 9 * * 1" (lunes 9:00)

CONTEXTS:
  ${{ github.actor }}          → usuario que disparó
  ${{ github.ref_name }}       → nombre de la rama
  ${{ github.sha }}            → SHA del commit
  ${{ inputs.mi_input }}       → input del workflow_dispatch
  ${{ env.MI_VAR }}            → variable de entorno
  ${{ secrets.API_KEY }}       → secreto (enmascarado)
  ${{ steps.X.outputs.Y }}     → output de un step
  ${{ needs.jobA.outputs.Y }}  → output de otro job (requiere needs:)
  ${{ matrix.os }}             → valor actual de la matriz

CLAVES ÚTILES EN STEPS:
  run:       comando shell
  uses:      reutilizar una Action
  if:        condicional (salta el step si false)
  id:        para referenciar outputs
  with:      parámetros de una action (uses:)
  env:       env vars para el step
  needs:     depende de otro job
  continue-on-error: true    → sigue aunque falle
  timeout-minutes: 30

EXPORTAR OUTPUTS:
  echo "mi_var=valor" >> $GITHUB_OUTPUT

EXPORTAR ENV PARA SIGUIENTES STEPS:
  echo "MI_VAR=valor" >> $GITHUB_ENV

DEBUG:
  Añade el secreto ACTIONS_STEP_DEBUG = true   → logs verbosos
'''
print(cheat)

## 18. 🎓 Siguiente paso

1. **Crea una rama** y prueba a escribir tu propio workflow:
   ```bash
   git checkout -b mi-primer-workflow
   # edita .github/workflows/mi_workflow.yaml
   git add .github/workflows/mi_workflow.yaml
   git commit -m "feat: mi primer workflow"
   git push origin mi-primer-workflow
   ```

2. **Ve a GitHub** → pestaña *Actions* → ejecuta el workflow.
3. **Lee los logs**. Si falla, el log te dice exactamente en qué step y qué comando.
4. **Itera**: arregla el YAML y vuelve a hacer push.

📚 Docs oficiales: https://docs.github.com/actions  
🧰 Marketplace de Actions: https://github.com/marketplace?type=actions